In [1]:
# Setup Spark SQL
# Note if running locally you need the JVM https://www.oracle.com/java/technologies/downloads/
# Consider running in https://colab.research.google.com/
%pip install pyspark

In [2]:
# Initialize Context - this is where you'd setup information about your Hadoop cluster if you had one!
from pyspark.sql import SparkSession


spark = SparkSession.builder.appName("Covid").getOrCreate()

sc = spark.sparkContext

sc.setLogLevel("WARN")

In [3]:
# Download 100mb covid county data file
!curl "https://raw.githubusercontent.com/nytimes/covid-19-data/master/us-counties.csv" > ./uscounties.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 99.9M  100 99.9M    0     0  28.8M      0  0:00:03  0:00:03 --:--:-- 28.8M


In [4]:
# Read the file into a Spark DataFrame
usCountiesFilePath = "./uscounties.csv"

df = spark.read.csv(usCountiesFilePath, inferSchema=True, header=True)

df.show()

+----------+-----------+----------+-----+-----+------+
|      date|     county|     state| fips|cases|deaths|
+----------+-----------+----------+-----+-----+------+
|2020-01-21|  Snohomish|Washington|53061|    1|     0|
|2020-01-22|  Snohomish|Washington|53061|    1|     0|
|2020-01-23|  Snohomish|Washington|53061|    1|     0|
|2020-01-24|       Cook|  Illinois|17031|    1|     0|
|2020-01-24|  Snohomish|Washington|53061|    1|     0|
|2020-01-25|     Orange|California| 6059|    1|     0|
|2020-01-25|       Cook|  Illinois|17031|    1|     0|
|2020-01-25|  Snohomish|Washington|53061|    1|     0|
|2020-01-26|   Maricopa|   Arizona| 4013|    1|     0|
|2020-01-26|Los Angeles|California| 6037|    1|     0|
|2020-01-26|     Orange|California| 6059|    1|     0|
|2020-01-26|       Cook|  Illinois|17031|    1|     0|
|2020-01-26|  Snohomish|Washington|53061|    1|     0|
|2020-01-27|   Maricopa|   Arizona| 4013|    1|     0|
|2020-01-27|Los Angeles|California| 6037|    1|     0|
|2020-01-2

In [5]:
# SparkSQL API
df.createOrReplaceTempView("covid")  # create table that you can do sql on

print("Max deaths:")
spark.sql(
    """
    select county, state, deaths
    from covid
    order by deaths desc
    limit 1
  """
).show()

Max deaths:
+-------------+--------+------+
|       county|   state|deaths|
+-------------+--------+------+
|New York City|New York| 40267|
+-------------+--------+------+



In [6]:
# DataFrame style
from pyspark.sql.functions import col

print("Max deaths:")
print(
    df.orderBy(col("deaths").desc()).take(  # .where(col("county") == "New York City") \
        1
    )
)

Max deaths:
[Row(date=datetime.date(2022, 5, 13), county='New York City', state='New York', fips=None, cases=2422658, deaths=40267)]


In [7]:
# RDD MapReduce Style without key
rows = df.rdd


def getMax(cumm, other):
    if other["deaths"] is not None and other["deaths"] > cumm["deaths"]:
        return other
    else:
        return cumm


print("Max deaths:")
print(rows.reduce(getMax))

Max deaths:
Row(date=datetime.date(2022, 5, 13), county='New York City', state='New York', fips=None, cases=2422658, deaths=40267)


In [8]:
# RDD MapReduce Style with mapped tuples
rows = df.rdd


def getMax(cumm, other):
    if other[0] > cumm[0]:
        return other
    else:
        return cumm


rows = rows.map(lambda r: (r["deaths"] or 0, f"{r['county']},{r['state']}"))
print("Max deaths:")
print(rows.reduce(getMax))

Max deaths:
(40267, 'New York City,New York')


In [40]:
# Write code to find the county with the most deaths

df = spark.read.csv(usCountiesFilePath, inferSchema=True, header=True)

df.createOrReplaceTempView("covid")

print('County with Most Deaths: ')
spark.sql('''
Select county, deaths as Number_of_Deaths
From covid
Order By Number_of_Deaths Desc
Limit 1
''').show()


County with Most Deaths: 
+-------------+----------------+
|       county|Number_of_Deaths|
+-------------+----------------+
|New York City|           40267|
+-------------+----------------+



In [18]:
# Write code to find the total number of deaths in Utah county

df = spark.read.csv(usCountiesFilePath, inferSchema=True, header=True)
df.createOrReplaceTempView("covid")

print('total number of deaths in Utah County: ')

spark.sql('''
Select county, max(deaths) as Number_of_Deaths
From covid
Where county = 'Utah'
Group By county
''').show()


total number of deaths in Utah County: 
+------+----------------+
|county|Number_of_Deaths|
+------+----------------+
|  Utah|             791|
+------+----------------+



In [34]:
# Write code to find the county with the most cases

df = spark.read.csv(usCountiesFilePath, inferSchema=True, header=True)
df.createOrReplaceTempView("covid")

print('County with the most cases: ')
spark.sql('''
Select county, max(cases) as Number_of_Cases
From covid
Group By county
Order By Number_of_Cases desc
Limit 1
''').show()


County with the most cases: 
+-----------+---------------+
|     county|Number_of_Cases|
+-----------+---------------+
|Los Angeles|        2908425|
+-----------+---------------+



In [35]:
# Write code to find the death rate for each state and sort the states by death rate descending

df = spark.read.csv(usCountiesFilePath, inferSchema=True, header=True)
df.createOrReplaceTempView("covid")

print('Death Rate by State')
spark.sql('''
Select state, round((max(deaths)/max(cases)), 4) as death_rate
From covid
Group By state
Order By death_rate desc
''').show(100)



Death Rate by State
+--------------------+----------+
|               state|death_rate|
+--------------------+----------+
|         Puerto Rico|     0.063|
|            Michigan|    0.0191|
|            New York|    0.0166|
|        Pennsylvania|    0.0161|
|              Nevada|    0.0156|
|       West Virginia|    0.0152|
|          New Jersey|    0.0151|
|         Connecticut|    0.0147|
|            Missouri|    0.0147|
|         Mississippi|    0.0147|
|            Maryland|     0.014|
|           Tennessee|    0.0137|
|             Arizona|    0.0135|
|       Massachusetts|    0.0134|
|             Indiana|    0.0132|
|                Ohio|    0.0128|
|             Montana|    0.0128|
|             Alabama|    0.0128|
|            Illinois|    0.0125|
|          New Mexico|    0.0124|
|            Oklahoma|    0.0123|
|           Louisiana|    0.0117|
|      South Carolina|    0.0116|
|            Arkansas|    0.0114|
|        Rhode Island|    0.0111|
|             Wyoming|     0

In [38]:
# Write code to something else interesting with this data – your choice

# When did utah get its first covid case?

df = spark.read.csv(usCountiesFilePath, inferSchema=True, header=True)
df.createOrReplaceTempView("covid")

print('When did Utah get its first Covid Case?')
spark.sql('''
Select state, date
From covid
Where state = 'Utah' and cases = 1
Order By date
Limit 1
''').show()



When did Utah get its first Covid Case?
+-----+----------+
|state|      date|
+-----+----------+
| Utah|2020-02-25|
+-----+----------+

